In [1]:
# Data libraries

import pandas as pd
import numpy as np

In [ ]:
MATH112052026-7SS1SEM1/MLP-Project/reshape_function.ipynb

Data cleaning:

* Convert date headers to real dates
* Convert text labels (`County`, `Station_Name`, `Observation_ID`) as categories
* Reorder rows by `Observation_ID` then `date` 

In [2]:
def reshape_long(avg_df, min_df, max_df, loc_df):
    
    data = [avg_df, min_df, max_df]
    label = ["avg", "min", "max"]
    long_df = {}
    
    for df, lab in zip(data, label):
        long_df[lab] = df.melt(id_vars = ["County", "Station_Name", "Observation_ID"],
                               var_name = "date",
                               value_name = lab)
        long_df[lab]["date"] = pd.to_datetime(long_df[lab]["date"], format = "%d/%m/%Y")
        
    column = ["County", "Station_Name", "Observation_ID", "date"]
    
    river_long = (long_df["avg"]
                 .merge(long_df["min"], on = column, how = "left")
                 .merge(long_df["max"], on = column, how = "left")
                 .merge(loc_df, on = ["County", "Station_Name", "Observation_ID"], how = "left"))

    for col in ["County", "Station_Name", "Observation_ID"]:
        river_long[col] = river_long[col].astype("category")

    river_long = (river_long.sort_values(["Observation_ID", "date"]).reset_index(drop = True))

    return river_long

In [ ]:
# Load data

river_avg = pd.read_csv("data/avg_river_levels.csv")
river_min = pd.read_csv("data/min_river_levels.csv")
river_max = pd.read_csv("data/max_river_levels.csv")

river_location = pd.read_csv("data/monitoring_locations.csv")

In [ ]:
river_long = reshape_long(river_avg, river_min, river_max, river_location)
river_long.head()